# Agent-driven Training with the Training Gym CLI

The Training Gym gives developers more than an API for writing training
configurations. Its bundled skill teaches the agent how to own the full
RL training, monitoring, and debugging lifecycle.

This tutorial is a hands-on walkthrough of that loop. You will:

1. Install the Training Gym agent skills.
2. Give your agent a training objective.
3. Watch the agent build a Qwen3-4B training configuration.
4. See how the agent autonomously monitors the run and iterates with the CLI.

The training objective we'll use in this tutorial is to **post-train a model 
to answer in rhyme**.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main
if importlib.util.find_spec('nltk') is None:
    %uv pip install -q nltk

## 1. Introducing the CLI

The `training-gym` CLI is the operational interface for agent-driven
training. It installs the skills that teach an agent the training workflow
and provides commands for monitoring training runs through their status,
rewards, logs, and rollout traces.

The CLI is installed with the package. Start by looking at its top-level
command groups:

```bash
training-gym --help
```

In [ ]:
!training-gym --help

## 2. Install the agent skills

The bundled `agent-driven-training` skill tells the agent to:

- ask before making choices that change model behavior or GPU cost
- catch dataset and reward bugs locally before they waste GPU time
- scale up only after cheaper, shorter test runs show the training pipeline is healthy
- inspect actual model outputs to verify that reward reflects the intended behavior
- investigate suspicious reward trends so metric exploits are not mistaken for learning

Install the bundled skills into the current project:

```bash
training-gym skills install
```

In [ ]:
!training-gym skills install

You can just ask your agent:

> can you post-train a model to rhyme in its output

Given an ambiguous request, the agent will propose a model, dataset, reward,
and cluster shape, then ask you to confirm the choices
that affect behavior and cost. For the example session below, the agent
selected **Qwen3-4B**, `tatsu-lab/alpaca`, and a reward that grants rhyme
credit only when the response rhymes and remains relevant.

## 3. Review the code the agent generated

The Training Gym gives the agent the framework primitives and lifecycle
guidance; the result is ordinary Python that you can inspect, edit, and run.
The rest of this section is the rhyming configuration produced for the
request above.

The dataset keeps self-contained Alpaca instructions and uses each reference
answer to measure whether a rhyming response stayed on topic.

The agent combined a deterministic rhyme score with an embedding-based
relevance score. Relevance gates rhyme so unrelated verse cannot win, and
anti-reward hacking checks penalize repeated end words and one-word lines.
Before spending any GPU time, the agent exercised the reward on correct,
non-rhyming, off-topic, repeated-word, malformed, and empty responses.

The complete generated configuration is shown as one file below.

In [ ]:
import re

from modal_training_gym import HuggingFaceDataset, Qwen3_4B, TrainConfig
from modal_training_gym.train_recipes.slime_recipe import Qwen3_4b_Recipe

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIR = "/opt/rhyme-embedder"

SYSTEM_PROMPT = (
    "You are a poet who answers every question in rhyme. Answer the question "
    "correctly and completely, but write the entire answer as verse: at least "
    "four lines, one clause per line, with line endings that rhyme in couplets "
    "(AABB). Do not write any prose, preamble, or explanation outside the verse."
)


class RhymeInstructionDataset(HuggingFaceDataset):
    """Self-contained Alpaca instructions; the label is the reference answer.

    Rows carrying an extra ``input`` field are dropped — the prompt template
    only passes ``instruction``, so those rows would ask an unanswerable
    question. Reference answers are length-bounded to keep the embedding
    comparison meaningful (a two-word label has no topic to match).
    """

    hf_repo = "tatsu-lab/alpaca"
    input_column = "instruction"
    output_column = "output"
    output_format = "jsonl"
    apply_chat_template = True
    always_prepare = True
    system_prompt = SYSTEM_PROMPT
    prompt_template = "{input}"

    def load(self, split: str = "all"):
        from datasets import load_dataset

        ds = load_dataset(self.hf_repo, self.hf_config, split=self.hf_split)
        ds = ds.filter(
            lambda r: not r["input"].strip() and 60 <= len(r["output"]) <= 600
        )
        if self.n_rows:
            ds = ds.select(range(min(self.n_rows, len(ds))))
        return ds


_CMUDICT: dict = {}
_VOWELS = ("A", "E", "I", "O", "U")

def _cmudict() -> dict:
    if not _CMUDICT:
        import nltk
        from nltk.corpus import cmudict

        nltk.download("cmudict", quiet=True)
        _CMUDICT.update(cmudict.dict())
    return _CMUDICT

def _strip_thinking(text: str) -> str:
    """Drop a ``<think>`` block and any stray markdown bullets/numbering."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"</?think>", "", text)
    return text.strip()

def _lines(text: str) -> list[str]:
    return [
        line.strip() for line in _strip_thinking(text).split("\n") if line.strip()
    ]

def _end_word(line: str) -> str:
    words = re.findall(r"[a-zA-Z']+", line)
    return words[-1].lower().strip("'") if words else ""

def rhyme_tail(word: str) -> tuple:
    """Phonemes from the last stressed vowel onward, stress markers removed.

    Falls back to the last three letters for words the dictionary doesn't
    know (names, coinages), which is a decent orthographic proxy.
    """
    if not word:
        return ()
    phones = _cmudict().get(word)
    if not phones:
        return ("~", word[-3:])
    seq = phones[0]
    stressed = [i for i, p in enumerate(seq) if p[-1] in ("1", "2")]
    if stressed:
        start = stressed[-1]
    else:
        vowels = [i for i, p in enumerate(seq) if p[0] in _VOWELS]
        start = vowels[-1] if vowels else 0
    return tuple(re.sub(r"\d", "", p) for p in seq[start:])

def words_rhyme(a: str, b: str) -> bool:
    """True when two *different* words share a rhyme tail.

    A word never rhymes with itself — otherwise repeating one end word would
    score a perfect rhyme scheme.
    """
    if not a or not b or a == b:
        return False
    return rhyme_tail(a) == rhyme_tail(b)

def _scheme_score(end_words: list[str], offset: int) -> float:
    """Fraction of rhyming pairs: offset 1 = AABB, offset 2 = ABAB."""
    pairs = []
    for start in range(0, len(end_words) - offset, 2 * offset):
        for k in range(offset):
            i, j = start + k, start + k + offset
            if j < len(end_words):
                pairs.append((end_words[i], end_words[j]))
    if not pairs:
        return 0.0
    return sum(words_rhyme(a, b) for a, b in pairs) / len(pairs)

def score_rhyme(response: str) -> float:
    """Rhyme quality of a response in ``[0, 1]``.

    Combines the best-fitting rhyme scheme with two anti-gaming factors:
    the share of distinct end words, and the share of lines with real
    substance (three or more words).
    """
    lines = _lines(response)
    if len(lines) < 2:
        return 0.0
    end_words = [_end_word(line) for line in lines]
    if not any(end_words):
        return 0.0

    scheme = max(_scheme_score(end_words, 1), _scheme_score(end_words, 2))
    distinct = len({w for w in end_words if w}) / len(end_words)
    substantial = sum(
        len(re.findall(r"[a-zA-Z']+", line)) >= 3 for line in lines
    ) / len(lines)
    length_factor = min(1.0, len(lines) / 4)
    return scheme * distinct * substantial * length_factor


_EMBEDDER: dict = {}

def _embedder():
    """Mean-pooling MiniLM loaded once per worker from the baked image dir."""
    if not _EMBEDDER:
        import torch
        from transformers import AutoModel, AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(EMBED_DIR)
        model = AutoModel.from_pretrained(EMBED_DIR)
        model.eval()
        _EMBEDDER["tokenizer"] = tokenizer
        _EMBEDDER["model"] = model
        _EMBEDDER["torch"] = torch
        print(f"[rhyme_rm] embedder ready: {EMBED_MODEL}")
    return _EMBEDDER

def embed(texts: list[str]) -> list:
    """L2-normalized mean-pooled sentence embeddings."""
    parts = _embedder()
    torch = parts["torch"]
    batch = parts["tokenizer"](
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )
    with torch.no_grad():
        out = parts["model"](**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return torch.nn.functional.normalize(pooled, p=2, dim=1)

def _lexical_overlap(a: str, b: str) -> float:
    """Token-F1 fallback used only if the embedder fails to load."""
    ta = {w for w in re.findall(r"[a-z']+", a.lower()) if len(w) > 3}
    tb = {w for w in re.findall(r"[a-z']+", b.lower()) if len(w) > 3}
    if not ta or not tb:
        return 0.0
    return 2 * len(ta & tb) / (len(ta) + len(tb))

def score_relevance(response: str, reference: str) -> float:
    """Topical agreement with the reference answer, rescaled to ``[0, 1]``."""
    response = _strip_thinking(response)
    reference = (reference or "").strip()
    if not response or not reference:
        return 0.0
    try:
        vectors = embed([response, reference])
        cosine = float((vectors[0] * vectors[1]).sum())
    except Exception as exc:  # noqa: BLE001
        print(f"[rhyme_rm] embedder unavailable ({exc}); using lexical overlap")
        cosine = _lexical_overlap(response, reference)
    return max(0.0, min(1.0, (cosine - 0.10) / 0.45))


def rhyme_reward(response: str, reference: str) -> float:
    """Gated rhyme quality plus a smaller standalone relevance term."""
    rhyme = score_rhyme(response)
    relevance = score_relevance(response, reference)
    gate = min(1.0, relevance / 0.4)
    return gate * rhyme + 0.3 * relevance

async def rhyme_rm(args, sample, **kwargs) -> float:
    model = Qwen3_4B()
    response = model.parse_response(getattr(sample, "response", "") or "")
    reference = getattr(sample, "label", "") or ""
    return rhyme_reward(response.content or "", str(reference))


def _image_overlay(image):
    return image.run_commands(
        "uv pip install --system 'nltk>=3.8.0'",
        "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        # Download through a scratch cache so the image does not leave
        # files where the shared Hugging Face Volume needs to mount.
        "HF_HOME=/tmp/hf-build HF_HUB_CACHE=/tmp/hf-build "
        'python -c "from huggingface_hub import snapshot_download; '
        f"snapshot_download('{EMBED_MODEL}', local_dir='{EMBED_DIR}')\"",
        "rm -rf /tmp/hf-build /root/.cache/huggingface",
    )

def build_config(
    *, num_rollout: int, n_rows: int, save_interval: int
) -> TrainConfig:
    return TrainConfig(
        model=Qwen3_4B(),
        dataset=RhymeInstructionDataset(n_rows=n_rows),
        recipe=Qwen3_4b_Recipe(
            custom_rm_function=rhyme_rm,
            num_rollout=num_rollout,
            rollout_batch_size=16,
            n_samples_per_prompt=8,
            rollout_max_response_len=1024,
            rollout_temperature=1.0,
            save_interval=save_interval,
            eval_interval=None,
            apply_chat_template_kwargs='{"enable_thinking": false}',
            capture_trace=True,
            trace_sample_limit=16,
            image_overlay=_image_overlay,
        ),
    )

## 4. Launch the one-step proof

Start with a one-step proof (`num_rollout=1`) to catch configuration, data,
and reward failures before committing to a longer run. `launch()` starts a
detached Modal app and returns the run ID the agent uses to monitor it with
the CLI.

In [ ]:
training_run = build_config(num_rollout=1, n_rows=512, save_interval=1)
run = training_run.launch()
RUN_ID = run.training_run_id
print(f"training_run_id: {RUN_ID}")

## 5. Inspect the run with the CLI

After writing the initial scaffolding, the agent uses the CLI to monitor the
training run.

First, verify that the run was recorded. `run list` is also how you recover
an ID after closing a terminal or notebook.

```bash
training-gym run list --since 2h
```

In [ ]:
!training-gym run list --since 2h

Use `run get --verbose` as the normal progress check. It reports the current
stage and step plus reward history and rollout summaries.

```bash
training-gym run get <run-id> --verbose
```

In [ ]:
!training-gym run get {RUN_ID} --verbose

Inspect logs when `run get` reports a failure or its step stops
advancing.

```bash
training-gym run logs <run-id>
```

In [ ]:
!training-gym run logs {RUN_ID}

Reward is only a proxy. Before promoting the proof, download its rollout
traces and read the actual prompts, responses, and per-sample rewards. The
dry run previews the download; the second command writes traces beneath
`./traces/<run-id>/`.

```bash
training-gym run trace <run-id> --out ./traces --dry-run
training-gym run trace <run-id> --out ./traces --yes
```

In [ ]:
!training-gym run trace {RUN_ID} --out ./traces --dry-run
!training-gym run trace {RUN_ID} --out ./traces --yes

The agent uses the same sequence at every scale:

1. **Prove one step:** require one completed rollout, nonempty reward, no
   traceback, and sensible samples.
2. **Smoke test:** launch about ten steps and inspect the trajectory plus
   baseline, transition, and recent traces.
3. **Promote:** launch the full horizon only when the signal is informative
   and the responses are genuinely improving.

Repeat `run get`, `run logs`, and `run trace` with each new run ID. The
agent can consume `--json` output when monitoring automatically; the
human-readable output used here is easier to learn from.

## 6. Watch the agent catch reward hacking

The timelapsed demo below shows the loop in practice. A rising reward initially looked
healthy, but trace inspection exposed responses gaming the rhyme signal with
repeated end words and undersized lines. The agent tightened the scorer,
reran the cheap stage, and checked real samples again before promotion.

<video controls playsinline width="100%">
  <source src="/agent-driven-training-rhyme.mp4" type="video/mp4">
  <source src="https://gym.modal.dev/agent-driven-training-rhyme.mp4" type="video/mp4">
  <a href="https://gym.modal.dev/agent-driven-training-rhyme.mp4">Watch the agent-driven training demo.</a>
</video>

## 7. Results

The full run finished in 46 minutes, and the numbers tell the story:

- Rhyme score: 0.475 → 0.908
- Answers above the rhyme threshold: 27% → 84%
- Non-rhyming answers: 35/128 → 0/128
- Relevance: 0.877 → 0.886 (essentially flat)

That last line is the one that matters. Relevance held steady while rhyme 
climbed, which is the evidence that the model learned to rhyme *in addition 
to* answering the question as intended.

## What to take away

Installing the Training Gym skill gives your agent a runbook for the full
training lifecycle. Instead of stopping after it generates a configuration,
the agent knows to preflight locally, prove one step, smoke test, diagnose
problems, and promote only when a run is healthy.

The CLI provides the observability that unlocks the agent's ability to make 
those training judgments. Run status and logs distinguish slow startup from 
failure, reward history shows whether learning is occurring, and rollout traces 
reveal whether the model is improving or merely gaming the reward.

Together, the skill provides the learning judgment and the CLI provides the evidence:
install the skill, then let the CLI's observability guide each training 
decision.